# 11 - Run a scheduled training experiment

Notebook 10 showed that more training steps slightly reduced validation loss. This notebook continues from the best saved checkpoint with a lower learning rate that decays gradually. It compares validation loss and one fixed greedy generation before and after the run. Only after checkpoint selection is complete does it score the reserved test split once.

The experiment starts from the latest checkpoint and keeps the best validation state. Test documents are not read for training or checkpoint selection; the final test score is written after the selected checkpoint has been saved.

## 1. Load the checkpoint and unchanged evaluation data

The saved tokenizer checksum and model configuration must match. A mismatch means the vocabulary or architecture changed, so this checkpoint cannot be resumed safely.

In [1]:
import copy
import hashlib
import json
import math
import random
from dataclasses import dataclass
from pathlib import Path

import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tokenizers import Tokenizer

SEED = 123
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def find_project_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook from the darija-llm-lab project directory.")

PROJECT_ROOT = find_project_root()
TOKENIZER_PATH = PROJECT_ROOT / "artifacts/tokenizer/tokenizer_darija.json"
TRAIN_PATH = PROJECT_ROOT / "data/processed/train.jsonl"
VALIDATION_PATH = PROJECT_ROOT / "data/processed/validation.jsonl"
TEST_PATH = PROJECT_ROOT / "data/processed/test.jsonl"
TOKENIZER_MANIFEST_PATH = PROJECT_ROOT / "artifacts/tokenizer/training_manifest.json"
CHECKPOINT_PATH = PROJECT_ROOT / "artifacts/checkpoints/darija_gpt_best.pt"
required = [TOKENIZER_PATH, TRAIN_PATH, VALIDATION_PATH, TEST_PATH, TOKENIZER_MANIFEST_PATH, CHECKPOINT_PATH]
missing = [str(path.relative_to(PROJECT_ROOT)) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "Required files are missing: " + ", ".join(missing) + ". Run notebooks 01, 02, and 08 first."
    )

def read_documents(path):
    documents = []
    with path.open(encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, start=1):
            if not line.strip():
                continue
            record = json.loads(line)
            text = record.get("text") if isinstance(record, dict) else None
            if not isinstance(text, str) or not text.strip():
                raise ValueError(f"{path.name}, line {line_number}: invalid text field.")
            documents.append(text)
    if not documents:
        raise ValueError(f"{path.name} contains no documents.")
    return documents

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
tokenizer.encode_special_tokens = True
eos_token_id = tokenizer.token_to_id("[EOS]")
if eos_token_id is None:
    raise ValueError("The tokenizer does not contain [EOS].")
tokenizer_sha256 = hashlib.sha256(TOKENIZER_PATH.read_bytes()).hexdigest()
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
if checkpoint["tokenizer_sha256"] != tokenizer_sha256:
    raise ValueError("The checkpoint uses a different tokenizer.")
expected_dataset_hashes = {
    "train": hashlib.sha256(TRAIN_PATH.read_bytes()).hexdigest(),
    "validation": hashlib.sha256(VALIDATION_PATH.read_bytes()).hexdigest(),
}
if checkpoint.get("dataset_sha256") != expected_dataset_hashes:
    raise ValueError("The checkpoint uses different train/validation splits. Rerun notebooks 01, 02, 08, and 10.")
tokenizer_manifest = json.loads(TOKENIZER_MANIFEST_PATH.read_text(encoding="utf-8"))
actual_test_hash = hashlib.sha256(TEST_PATH.read_bytes()).hexdigest()
if tokenizer_manifest.get("test_jsonl_sha256") != actual_test_hash:
    raise ValueError("The test split differs from the split recorded during tokenizer preparation.")
if "optimizer_state_dict" not in checkpoint:
    raise ValueError("Run the updated notebooks 08 and 10 to create a resumable checkpoint.")

train_documents = read_documents(TRAIN_PATH)
validation_documents = read_documents(VALIDATION_PATH)
print("Device:", device)
print("Checkpoint step:", checkpoint["best_step"])
print("Previous best validation loss:", checkpoint["training"]["best_validation_loss"])
print("Training / validation documents:", len(train_documents), "/", len(validation_documents))
print("Reserved test documents:", tokenizer_manifest["test_documents"])

Device: cpu
Checkpoint step: 300
Previous best validation loss: 6.699598477649619
Training / validation documents: 2369 / 296
Reserved test documents: 296


## 2. Restore the GPT classes and weights

The class structure and parameter names match the model saved by notebook 08. The model dimensions are read from that checkpoint rather than guessed again.

In [2]:
@dataclass(frozen=True)
class GPTConfig:
    vocab_size: int
    context_length: int
    embedding_dim: int
    num_layers: int
    num_heads: int
    dropout: float
    qkv_bias: bool = False


class MultiHeadCausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.num_heads = config.num_heads
        self.head_dim = config.embedding_dim // config.num_heads
        self.embedding_dim = config.embedding_dim
        self.query = nn.Linear(config.embedding_dim, config.embedding_dim, bias=config.qkv_bias)
        self.key = nn.Linear(config.embedding_dim, config.embedding_dim, bias=config.qkv_bias)
        self.value = nn.Linear(config.embedding_dim, config.embedding_dim, bias=config.qkv_bias)
        self.output = nn.Linear(config.embedding_dim, config.embedding_dim)
        self.attention_dropout = nn.Dropout(config.dropout)
        future_mask = torch.triu(
            torch.ones(config.context_length, config.context_length, dtype=torch.bool), diagonal=1
        )
        self.register_buffer("future_mask", future_mask)

    def forward(self, x):
        batch_size, sequence_length, _ = x.shape
        shape = (batch_size, sequence_length, self.num_heads, self.head_dim)
        query = self.query(x).view(shape).transpose(1, 2)
        key = self.key(x).view(shape).transpose(1, 2)
        value = self.value(x).view(shape).transpose(1, 2)
        scores = (query @ key.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(
            self.future_mask[:sequence_length, :sequence_length], -torch.inf
        )
        weights = self.attention_dropout(torch.softmax(scores, dim=-1))
        context = weights @ value
        context = context.transpose(1, 2).contiguous().view(
            batch_size, sequence_length, self.embedding_dim
        )
        return self.output(context)


class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(config.embedding_dim, 4 * config.embedding_dim),
            nn.GELU(),
            nn.Linear(4 * config.embedding_dim, config.embedding_dim),
        )

    def forward(self, x):
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.norm1 = nn.LayerNorm(config.embedding_dim)
        self.attention = MultiHeadCausalSelfAttention(config)
        self.norm2 = nn.LayerNorm(config.embedding_dim)
        self.feed_forward = FeedForward(config)
        self.residual_dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = x + self.residual_dropout(self.attention(self.norm1(x)))
        x = x + self.residual_dropout(self.feed_forward(self.norm2(x)))
        return x


class GPTModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.embedding_dim)
        self.position_embedding = nn.Embedding(config.context_length, config.embedding_dim)
        self.embedding_dropout = nn.Dropout(config.dropout)
        self.blocks = nn.Sequential(*(TransformerBlock(config) for _ in range(config.num_layers)))
        self.final_norm = nn.LayerNorm(config.embedding_dim)
        self.output_head = nn.Linear(config.embedding_dim, config.vocab_size, bias=False)
        self.apply(self._initialize_weights)

    @staticmethod
    def _initialize_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, token_ids):
        batch_size, sequence_length = token_ids.shape
        if sequence_length > self.config.context_length:
            raise ValueError("Input sequence exceeds context length.")
        positions = torch.arange(sequence_length, device=token_ids.device)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)[None, :, :]
        x = self.embedding_dropout(x)
        return self.output_head(self.final_norm(self.blocks(x)))


config = GPTConfig(**checkpoint["model_config"])
model = GPTModel(config).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
print("Model parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Context length:", config.context_length)

Model parameters: 359,808
Context length: 64


## 3. Rebuild training and validation windows

Use the same per-document EOS handling, context length, and stride. This keeps the new validation score comparable with the previous checkpoint.

In [3]:
class DarijaGPTDataset(Dataset):
    def __init__(self, documents, tokenizer, context_length, stride, eos_token_id):
        self.examples = []
        for document in documents:
            token_ids = tokenizer.encode(document).ids + [eos_token_id]
            for start in range(0, len(token_ids) - context_length, stride):
                window = token_ids[start:start + context_length + 1]
                if len(window) == context_length + 1:
                    self.examples.append(torch.tensor(window, dtype=torch.long))
        if not self.examples:
            raise ValueError("No context windows were created.")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        window = self.examples[index]
        return window[:-1], window[1:]


class SlidingEvaluationDataset(Dataset):
    """Score each next-token target once while retaining overlapping context."""
    def __init__(self, documents, tokenizer, context_length, stride, eos_token_id):
        if context_length < 2 or stride < 1 or stride > context_length:
            raise ValueError("Use a positive stride no larger than context_length.")
        self.examples = []
        pad_token_id = tokenizer.token_to_id("[PAD]")
        if pad_token_id is None:
            raise ValueError("The tokenizer must contain [PAD].")
        for document in documents:
            token_ids = tokenizer.encode(document).ids + [eos_token_id]
            covered_until = 0
            start = 0
            while start < len(token_ids) - 1:
                end = min(start + context_length + 1, len(token_ids))
                inputs = token_ids[start:end - 1]
                targets = token_ids[start + 1:end]
                labels = [target if start + offset + 1 > covered_until else -100
                          for offset, target in enumerate(targets)]
                pad_count = context_length - len(inputs)
                inputs += [pad_token_id] * pad_count
                labels += [-100] * pad_count
                if any(label != -100 for label in labels):
                    self.examples.append((
                        torch.tensor(inputs, dtype=torch.long),
                        torch.tensor(labels, dtype=torch.long),
                    ))
                covered_until = max(covered_until, end - 1)
                if end == len(token_ids):
                    break
                start += stride
        if not self.examples:
            raise ValueError("No evaluation targets were created.")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


CONTEXT_LENGTH = config.context_length
BATCH_SIZE = 8
WINDOW_STRIDE = CONTEXT_LENGTH // 2
train_dataset = DarijaGPTDataset(
    train_documents, tokenizer, CONTEXT_LENGTH, WINDOW_STRIDE, eos_token_id
)
validation_dataset = SlidingEvaluationDataset(
    validation_documents, tokenizer, CONTEXT_LENGTH, WINDOW_STRIDE, eos_token_id
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
validation_loader = DataLoader(
    validation_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)
print("Training windows:", len(train_dataset))
print("Validation windows:", len(validation_dataset))

Training windows: 33275
Validation windows: 4411


## 4. Define a resumable cosine-decay run

The optimizer resumes its saved AdamW moments but uses a smaller learning rate for this experiment. Cosine decay gradually lowers that rate from the starting value to the minimum. Validation loss selects the checkpoint; the training loss is not used for model selection.

In [4]:
class ScheduledTrainingRun:
    def __init__(self, model, train_loader, validation_loader, checkpoint, device, start_learning_rate):
        self.model = model
        self.train_loader = train_loader
        self.validation_loader = validation_loader
        self.device = device
        self.optimizer = torch.optim.AdamW(model.parameters(), lr=start_learning_rate, weight_decay=0.1)
        self.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        for group in self.optimizer.param_groups:
            group["lr"] = start_learning_rate
        self.global_step = checkpoint["best_step"]
        self.best_step = checkpoint["best_step"]
        self.best_validation_loss = checkpoint["training"]["best_validation_loss"]
        self.best_model_state = copy.deepcopy(checkpoint["model_state_dict"])
        self.best_optimizer_state = copy.deepcopy(checkpoint["optimizer_state_dict"])

    @torch.no_grad()
    def evaluate(self, loader):
        was_training = self.model.training
        self.model.eval()
        total_loss = 0.0
        total_tokens = 0
        for inputs, targets in loader:
            inputs, targets = inputs.to(self.device), targets.to(self.device)
            logits = self.model(inputs)
            total_loss += F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1),
                reduction="sum",
                ignore_index=-100,
            ).item()
            total_tokens += targets.ne(-100).sum().item()
        self.model.train(was_training)
        if total_tokens == 0:
            raise ValueError("The evaluation loader produced no target tokens.")
        return total_loss / total_tokens

    def fit(self, additional_steps=200, eval_interval=50, min_learning_rate=2e-5):
        if additional_steps < 1 or eval_interval < 1:
            raise ValueError("Step count and evaluation interval must be positive.")
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer, T_max=additional_steps, eta_min=min_learning_rate
        )
        stop_step = self.global_step + additional_steps
        history = []

        while self.global_step < stop_step:
            saw_batch = False
            for inputs, targets in self.train_loader:
                saw_batch = True
                self.model.train()
                inputs, targets = inputs.to(self.device), targets.to(self.device)
                self.optimizer.zero_grad(set_to_none=True)
                logits = self.model(inputs)
                loss = F.cross_entropy(logits.flatten(0, 1), targets.flatten())
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.optimizer.step()
                scheduler.step()
                self.global_step += 1

                if self.global_step % eval_interval == 0 or self.global_step == stop_step:
                    validation_loss = self.evaluate(self.validation_loader)
                    row = {
                        "step": self.global_step,
                        "learning_rate": self.optimizer.param_groups[0]["lr"],
                        "validation_loss": validation_loss,
                    }
                    history.append(row)
                    print(
                        f"Step {self.global_step:>4} | lr {row['learning_rate']:.6g} "
                        f"| validation loss {validation_loss:.4f}"
                    )
                    if validation_loss < self.best_validation_loss:
                        self.best_validation_loss = validation_loss
                        self.best_step = self.global_step
                        self.best_model_state = {
                            key: value.detach().cpu().clone()
                            for key, value in self.model.state_dict().items()
                        }
                        self.best_optimizer_state = copy.deepcopy(self.optimizer.state_dict())

                if self.global_step >= stop_step:
                    break
            if not saw_batch:
                raise ValueError("The training loader produced no batches.")

        self.model.load_state_dict(self.best_model_state)
        self.optimizer.load_state_dict(self.best_optimizer_state)
        return history


START_LEARNING_RATE = 1e-4
runner = ScheduledTrainingRun(model, train_loader, validation_loader, checkpoint, device, START_LEARNING_RATE)
previous_loss = runner.evaluate(validation_loader)
print(f"Restored checkpoint loss: {previous_loss:.4f}")
print("Resume step:", runner.global_step)

Restored checkpoint loss: 6.6996
Resume step: 300


## 5. Compare one fixed prompt before and after training

Greedy decoding makes the comparison repeatable: each step selects the most likely next token. The prompt comes from the validation split, but it is used only for text generation and never for gradient updates.

In [5]:
@torch.no_grad()
def generate_greedy(model, tokenizer, prompt, context_length, eos_token_id, max_new_tokens=40):
    model.eval()
    token_ids = torch.tensor(
        [tokenizer.encode(prompt).ids], dtype=torch.long, device=device
    )
    for _ in range(max_new_tokens):
        context = token_ids[:, -context_length:]
        next_token = model(context)[:, -1, :].argmax(dim=-1, keepdim=True)
        token_ids = torch.cat((token_ids, next_token), dim=1)
        if next_token.item() == eos_token_id:
            break
    return tokenizer.decode(token_ids[0].tolist(), skip_special_tokens=True)


prompt = " ".join(validation_documents[0].split()[:8])
text_before = generate_greedy(
    model, tokenizer, prompt, config.context_length, eos_token_id
)
print("Prompt:", prompt)
print("Before:", text_before)

Prompt: ...وبيني وبينك انا شايفو وحاس بيه جابها يبرد
Before: ...وبيني وبينك انا شايفو وحاس بيه جابها يبرد........................................................................................................................


## 6. Train with cosine learning-rate decay

The run adds 200 optimizer steps, evaluates every 50 steps, and retains the best validation checkpoint. If the validation score does not improve, the original checkpoint remains the selected model.

In [6]:
ADDITIONAL_STEPS = 200
EVAL_INTERVAL = 50
MIN_LEARNING_RATE = 2e-5

history = runner.fit(
    additional_steps=ADDITIONAL_STEPS,
    eval_interval=EVAL_INTERVAL,
    min_learning_rate=MIN_LEARNING_RATE,
)
resumed_loss = runner.evaluate(validation_loader)
resumed_perplexity = math.exp(resumed_loss) if resumed_loss < 20 else float("inf")
text_after = generate_greedy(
    model, tokenizer, prompt, config.context_length, eos_token_id
)
print(f"Selected step: {runner.best_step}")
print(f"Validation loss: {resumed_loss:.4f}")
print(f"Validation perplexity: {resumed_perplexity:.2f}")
print("After:", text_after)

Step  350 | lr 8.82843e-05 | validation loss 6.6936


Step  400 | lr 6e-05 | validation loss 6.6887


Step  450 | lr 3.17157e-05 | validation loss 6.6819


Step  500 | lr 2e-05 | validation loss 6.6795


Selected step: 500
Validation loss: 6.6795
Validation perplexity: 795.89
After: ...وبيني وبينك انا شايفو وحاس بيه جابها يبرد... و و... و... و... و... و و... و... و و... و...... و... و و و... و و و...... و و و...... و... و


## 7. Compare results and save the experiment

The key comparison is validation loss on the same held-out split. Perplexity is also reported for readability. The two generated texts are qualitative checks and do not replace human review.

In [7]:
previous_best_loss = checkpoint["training"]["best_validation_loss"]
comparison = {
    "previous_best_step": checkpoint["best_step"],
    "previous_validation_loss": previous_best_loss,
    "scheduled_best_step": runner.best_step,
    "scheduled_validation_loss": resumed_loss,
    "scheduled_perplexity": resumed_perplexity,
    "validation_loss_improved": resumed_loss < previous_best_loss,
    "additional_steps": ADDITIONAL_STEPS,
    "schedule": {
        "name": "cosine",
        "start_learning_rate": START_LEARNING_RATE,
        "minimum_learning_rate": MIN_LEARNING_RATE,
        "evaluation_interval": EVAL_INTERVAL,
    },
    "prompt": prompt,
    "greedy_text_before": text_before,
    "greedy_text_after": text_after,
    "history": history,
}
print(json.dumps({
    key: value for key, value in comparison.items()
    if key not in {"greedy_text_before", "greedy_text_after", "history"}
}, indent=2))

if comparison["validation_loss_improved"]:
    checkpoint["model_state_dict"] = {
        key: value.detach().cpu() for key, value in model.state_dict().items()
    }
    checkpoint["optimizer_state_dict"] = runner.best_optimizer_state
    checkpoint["best_step"] = runner.best_step
    checkpoint["completed_steps"] = runner.global_step
    checkpoint["history"] = checkpoint.get("history", []) + history
    checkpoint["training"]["best_validation_loss"] = resumed_loss
    checkpoint["training"]["validation_perplexity"] = resumed_perplexity
    checkpoint["training"]["schedule"] = comparison["schedule"]
    torch.save(checkpoint, CHECKPOINT_PATH)
    print("Updated best checkpoint:", CHECKPOINT_PATH.relative_to(PROJECT_ROOT))
else:
    print("Kept the previous best checkpoint.")

comparison_path = PROJECT_ROOT / "artifacts/evaluation/cosine_training_comparison.json"
comparison_path.parent.mkdir(parents=True, exist_ok=True)
comparison_path.write_text(
    json.dumps(comparison, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print("Saved experiment report:", comparison_path.relative_to(PROJECT_ROOT))

# Final test evaluation: this is intentionally after all validation-based checkpoint decisions.
test_documents = read_documents(TEST_PATH)
test_dataset = SlidingEvaluationDataset(
    test_documents, tokenizer, CONTEXT_LENGTH, WINDOW_STRIDE, eos_token_id
)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loss = runner.evaluate(test_loader)
test_perplexity = math.exp(test_loss) if test_loss < 20 else float("inf")
test_target_tokens = sum(int(targets.ne(-100).sum()) for _, targets in test_dataset.examples)
test_record = {
    "checkpoint_best_step": runner.best_step,
    "test_documents": len(test_documents),
    "test_windows": len(test_dataset),
    "test_target_tokens": test_target_tokens,
    "test_loss": test_loss,
    "test_perplexity": test_perplexity,
    "targets_counted_once": True,
    "tokenizer_sha256": tokenizer_sha256,
    "test_jsonl_sha256": actual_test_hash,
}
test_report_path = PROJECT_ROOT / "artifacts/evaluation/final_test_evaluation.json"
test_report_path.write_text(json.dumps(test_record, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Final test loss:", f"{test_loss:.4f}")
print("Final test perplexity:", f"{test_perplexity:.2f}")
print("Test target tokens counted once:", test_target_tokens)
print("Saved final test report:", test_report_path.relative_to(PROJECT_ROOT))

# Save deterministic generations from the selected checkpoint for final human review.
final_generation_records = []
for document in validation_documents[:5]:
    sample_prompt = " ".join(document.split()[:8])
    sample_text = generate_greedy(
        model, tokenizer, sample_prompt, config.context_length, eos_token_id
    )
    continuation = sample_text[len(sample_prompt):].lstrip() if sample_text.startswith(sample_prompt) else sample_text
    final_generation_records.append({
        "prompt": sample_prompt,
        "generated_text": sample_text,
        "continuation": continuation,
        "decoding": "greedy",
        "checkpoint_best_step": runner.best_step,
    })
final_generations_path = PROJECT_ROOT / "artifacts/evaluation/generation_samples_final.jsonl"
final_generations_path.write_text(
    "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in final_generation_records),
    encoding="utf-8",
)
print("Saved final checkpoint generations:", final_generations_path.relative_to(PROJECT_ROOT))

{
  "previous_best_step": 300,
  "previous_validation_loss": 6.699598477649619,
  "scheduled_best_step": 500,
  "scheduled_validation_loss": 6.679460776384101,
  "scheduled_perplexity": 795.8898336938207,
  "validation_loss_improved": true,
  "additional_steps": 200,
  "schedule": {
    "name": "cosine",
    "start_learning_rate": 0.0001,
    "minimum_learning_rate": 2e-05,
    "evaluation_interval": 50
  },
  "prompt": "...\u0648\u0628\u064a\u0646\u064a \u0648\u0628\u064a\u0646\u0643 \u0627\u0646\u0627 \u0634\u0627\u064a\u0641\u0648 \u0648\u062d\u0627\u0633 \u0628\u064a\u0647 \u062c\u0627\u0628\u0647\u0627 \u064a\u0628\u0631\u062f"
}
Updated best checkpoint: artifacts\checkpoints\darija_gpt_best.pt
Saved experiment report: artifacts\evaluation\cosine_training_comparison.json


Final test loss: 6.6898
Final test perplexity: 804.17
Test target tokens counted once: 142171
Saved final test report: artifacts\evaluation\final_test_evaluation.json
Saved final checkpoint generations: artifacts\evaluation\generation_samples_final.jsonl


## Interpretation

A small or absent change after 200 more steps is useful information: the model may need more data, better coverage, a different tokenizer, or a more deliberate model-size experiment. Compare held-out metrics and the fixed prompt before changing several settings at once.